# Exercise 5: Finite Differenzen

Wärmeleitungsgleichung auf $x \in [0, 1]$ mit $u(0) = u(1) = 0$, äquidistantes Gitter mit $N$ inneren Punkten
$x_j = j\,\Delta x$, $\Delta x = 1/(N+1)$.

Die Funktionen stehen in `fd.py`, die Tests in `test_fd.py`, die Theorie in `THEORIE.md`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from fd import grid, laplacian, solve, solve_neumann

plt.rcParams["figure.dpi"] = 110

## 1 Stationäre Wärmeleitung $-u'' = f$

$f = \pi^2 \sin\pi x$, exakte Lösung $u = \sin\pi x$.

### 1a) 3-Punkte-Stern und Matrix $A$

$$-u''(x_j) \approx \frac{-u_{j-1} + 2u_j - u_{j+1}}{\Delta x^2}, \qquad
A = \frac{1}{\Delta x^2}\begin{pmatrix} 2 & -1 & & \\ -1 & 2 & \ddots & \\ & \ddots & \ddots & -1 \\ & & -1 & 2 \end{pmatrix}$$

Jede Zeile ist die Gleichung an einem Gitterpunkt, jede Spalte eine Unbekannte $u_j$.
In der ersten und letzten Zeile fehlt ein Nachbar, weil $u_0 = u_{N+1} = 0$ bekannt sind.

In [ ]:
N = 5
x, dx = grid(N)
A = laplacian(N)
print(f"dx = {dx:.4f},  1/dx^2 = {1 / dx**2:.0f}")
print("A * dx^2 =")
print((A.toarray() * dx**2).astype(int))
print(f"gespeicherte Einträge: {A.nnz} von {N * N}")

#### Die Matrix als Bild

Links das Besetzungsmuster (`spy`) für $N = 30$: nur die Hauptdiagonale und die beiden Nebendiagonalen sind belegt,
also $3N - 2$ von $N^2$ Einträgen. Rechts die Werte von $\Delta x^2 A$ für $N = 8$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8))

A30 = laplacian(30)
ax1.spy(A30, markersize=4)
ax1.set_title(f"Besetzung von $A$, $N = 30$ ({A30.nnz} von {30 * 30} Einträgen)")

N8 = 8
_, dx8 = grid(N8)
M = laplacian(N8).toarray() * dx8**2
im = ax2.imshow(M, cmap="RdBu_r", vmin=-2, vmax=2)
for i in range(N8):
    for j in range(N8):
        if M[i, j] != 0:
            ax2.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center", fontsize=9,
                     color="white" if abs(M[i, j]) > 1.5 else "black")
ax2.set_xticks(range(N8), [f"$u_{{{j + 1}}}$" for j in range(N8)])
ax2.set_yticks(range(N8), [f"$x_{{{i + 1}}}$" for i in range(N8)])
ax2.set_xlabel("Spalte = Unbekannte")
ax2.set_ylabel("Zeile = Gleichung am Gitterpunkt")
ax2.set_title(r"$\Delta x^2 A$ für $N = 8$")
fig.colorbar(im, ax=ax2, shrink=0.8)
fig.tight_layout()
plt.show()

#### $A$ prüfen, bevor man löst

Eigenschaften, die man ohne Lösung kennt (Details in `THEORIE.md`, 1a):

In [ ]:
N = 20
x, dx = grid(N)
A = laplacian(N)

sym = abs(A - A.T).max()
sums = np.asarray(A.sum(axis=1)).ravel()
quad = A @ (x * (1 - x))
k = np.arange(1, N + 1)
lam = 4 / dx**2 * np.sin(k * np.pi * dx / 2) ** 2
eig = np.sort(np.linalg.eigvalsh(A.toarray()))

print(f"Symmetrie        max|A - A^T|            = {sym}")
print(f"Zeilensummen     innen max|.|             = {np.abs(sums[1:-1]).max():.2e}")
print(f"                 Rand * dx^2              = {sums[0] * dx**2:.3f}, {sums[-1] * dx**2:.3f}")
print(f"Polynom x(1-x)   max|A u - 2|             = {np.abs(quad - 2).max():.2e}")
print(f"Eigenwerte       max rel. Abw. zur Formel = {np.max(np.abs(eig - lam) / lam):.2e}")
print(f"positiv definit  kleinster Eigenwert      = {eig[0]:.4f}  (pi^2 = {np.pi**2:.4f})")

#### Lösen und mit der exakten Lösung vergleichen

In [ ]:
f = lambda x: np.pi**2 * np.sin(np.pi * x)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
xf = np.linspace(0, 1, 400)
ax1.plot(xf, np.sin(np.pi * xf), "k-", lw=1, label=r"exakt $\sin\pi x$")
for N in [3, 7, 15]:
    x, u = solve(f, N)
    ax1.plot(x, u, "o", ms=4, label=f"$N = {N}$")
    ax2.plot(x, u - np.sin(np.pi * x), "o-", ms=3, label=f"$N = {N}$")
    print(f"N = {N:3d}:  max Fehler = {np.abs(u - np.sin(np.pi * x)).max():.3e},"
          f"  pi^2 dx^2 / 12 = {np.pi**2 * grid(N)[1]**2 / 12:.3e}")
ax1.set_xlabel("$x$"); ax1.set_ylabel("$u$"); ax1.legend(); ax1.set_title("Lösung")
ax2.set_xlabel("$x$"); ax2.set_ylabel(r"$u_j - \sin\pi x_j$"); ax2.legend(); ax2.set_title("Fehler")
fig.tight_layout()
plt.show()

> **Frage:** Wie kann man $A$ prüfen, bevor man etwas löst?
>
> **Antwort:** Mit Eigenschaften, die man vorher kennt: $A$ ist symmetrisch, $\Delta x^2 A$ hat genau den Stern $(-1, 2, -1)$,
> die inneren Zeilensummen sind 0 (die zweite Ableitung einer Konstanten verschwindet), und für $x(1-x)$ ist $A u = 2$
> exakt, weil der Abbruchfehler $\frac{\Delta x^2}{12}u''''$ dort null ist. Die Eigenwerte stimmen mit
> $\lambda_k = \frac{4}{\Delta x^2}\sin^2\frac{k\pi\Delta x}{2}$ überein und sind alle positiv. Alle diese Prüfungen
> stehen auch als Tests in `test_fd.py`.

### 1b) Fehler über $\Delta x$

Gitterkonvergenzstudie: dasselbe Problem auf immer feineren Gittern $N = 2^k - 1$, also $\Delta x = 2^{-k}$,
von $N = 3$ bis $N \approx 10^6$. Für jedes Gitter der maximale Fehler gegen $\sin\pi x$.
Mit $N = 2^k - 1$ sind $\Delta x$, alle $x_j$ und $1/\Delta x^2$ exakt als `float64` darstellbar.

In [ ]:
f = lambda x: np.pi**2 * np.sin(np.pi * x)
ks = np.arange(2, 21)
Ns = 2**ks - 1
dxs = 1 / (Ns + 1)
errs = np.empty(len(Ns))
for i, N in enumerate(Ns):
    x, u = solve(f, N)
    errs[i] = np.max(np.abs(u - np.sin(np.pi * x)))

best = np.argmin(errs)
fit = ks <= 13  # only the range where truncation dominates
slope = np.polyfit(np.log(dxs[fit]), np.log(errs[fit]), 1)[0]
print(f"Steigung für dx >= {dxs[fit].min():.1e}:  {slope:.3f}")
print(f"kleinster Fehler {errs[best]:.2e} bei N = {Ns[best]}, dx = {dxs[best]:.2e}")
print()
print("    N          dx        Fehler     Fehler(dx)/Fehler(dx/2)")
for i, N in enumerate(Ns):
    ratio = f"{errs[i] / errs[i + 1]:8.2f}" if i + 1 < len(Ns) else ""
    print(f"{N:8d}  {dxs[i]:.3e}  {errs[i]:.3e}  {ratio}")

In [ ]:
eps = np.finfo(float).eps
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.loglog(dxs, errs, "o-", ms=4, label="gemessen")
ax.loglog(dxs, np.pi**2 * dxs**2 / 12, "k--", lw=1, label=r"$\pi^2\Delta x^2/12$ (Abbruchfehler)")
ax.loglog(dxs, eps / dxs**2, "k:", lw=1, label=r"$\varepsilon/\Delta x^2$ (grobe Rundungsschranke)")
ax.axvline(dxs[best], color="gray", lw=0.8)
ax.set_xlabel(r"$\Delta x$")
ax.set_ylabel(r"$\max_j |u_j - \sin\pi x_j|$")
ax.set_ylim(1e-13, 1)
ax.set_title("1b: Fehler über $\\Delta x$, $N = 3 \\dots 2^{20}-1$")
ax.legend()
fig.tight_layout()
plt.show()

> **Frage:** Welche Steigung erwartet man, welche bekommt man?
>
> **Antwort:** Der 3-Punkte-Stern hat den Abbruchfehler $\frac{\Delta x^2}{12}u''''$, erwartet ist also Steigung 2.
> Gemessen: Steigung 2,00 bis $N = 8191$, der Fehler viertelt sich pro Halbierung von $\Delta x$ und liegt fast genau
> auf $\pi^2\Delta x^2/12$.
>
> **Frage:** Unter welchem $\Delta x$ wächst der Fehler wieder, und warum?
>
> **Antwort:** Das Minimum liegt bei $\Delta x \approx 1{,}5\cdot10^{-5}$ ($N = 65535$) mit etwa $2\cdot10^{-10}$.
> Darunter wächst der Fehler unregelmäßig und steil an, bis $6\cdot10^{-7}$ bei $N \approx 10^6$. Ursache sind
> Rundungsfehler: $A$ hat Einträge $\sim 1/\Delta x^2$, im Stern ziehen sich fast gleich große Zahlen voneinander ab, und die
> Konditionszahl $\kappa(A) \approx 4/(\pi^2\Delta x^2)$ wächst. Die Schranke $\varepsilon/\Delta x^2$ ist eine
> Abschätzung für den schlimmsten Fall. Der echte Rundungsfehler liegt deutlich darunter, auch weil die Matrix bei
> $\Delta x = 2^{-k}$ exakt ist und nur das Lösen rundet. Ab dem Minimum bringt ein feineres Gitter also nichts mehr.

### 1c) Stencils höherer Ordnung

Zentrale Sterne für $-u''$ (Gewichte mal $1/\Delta x^2$, in `fd.STENCILS`):

$$\text{Ordnung 2: } (-1,\ 2,\ -1), \qquad
\text{Ordnung 4: } \tfrac{1}{12}(1,\ -16,\ 30,\ -16,\ 1), \qquad
\text{Ordnung 6: } \tfrac{1}{180}(-2,\ 27,\ -270,\ 490,\ -270,\ 27,\ -2)$$

Am Rand reicht der breitere Stern über $x_0 = 0$ hinaus. Zwei Varianten in `laplacian(N, order, boundary)`:

- `"reduce"`: In den randnahen Zeilen den größten zentralen Stern nehmen, der noch passt (Zeile 1 mit 3 Punkten, bei Ordnung 6 Zeile 2 mit 5 Punkten).
- `"mirror"`: Geisterpunkte per ungerader Spiegelung, $u_{-m} = -u_m$. Das ist richtig, wenn alle geraden Ableitungen am Rand verschwinden, wie bei $\sin\pi x$.

In [ ]:
print("Delta x^2 * A, Ordnung 4, N = 7:")
for b in ["reduce", "mirror"]:
    print(f"\n{b}:")
    print(np.round(laplacian(7, 4, b).toarray() * grid(7)[1]**2 * 12, 3).astype(int), " (mal 1/12)")

In [ ]:
ks = np.arange(2, 17)
Ns = 2**ks - 1
dxs = 1 / (Ns + 1)
variants = [(2, "reduce"), (4, "reduce"), (4, "mirror"), (6, "reduce"), (6, "mirror")]
errs_c = {}
for order, b in variants:
    e = []
    for N in Ns:
        x, u = solve(f, N, order, b)
        e.append(np.max(np.abs(u - np.sin(np.pi * x))))
    errs_c[order, b] = np.array(e)

print("Rate log2(e(dx)/e(dx/2)) für N = 7 ... 255, Minimum des Fehlers:")
for (order, b), e in errs_c.items():
    rates = np.log2(e[:-1] / e[1:])[1:7]
    i = np.argmin(e)
    print(f"  Ordnung {order}, {b:6s}:  " + " ".join(f"{r:5.2f}" for r in rates)
          + f"   min {e[i]:.1e} bei N = {Ns[i]}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
styles = {"reduce": "o-", "mirror": "s--"}
for (order, b), e in errs_c.items():
    ax1.loglog(dxs, e, styles[b], ms=4, label=f"Ordnung {order}, {b}")
for p in [2, 4, 6]:
    ax1.loglog(dxs[:6], 1e-3 * (dxs[:6] / dxs[0])**p, "k-", lw=0.6)
    ax1.text(dxs[5], 1e-3 * (dxs[5] / dxs[0])**p, f"  $\\Delta x^{p}$", va="center", fontsize=8)
ax1.set_xlabel(r"$\Delta x$")
ax1.set_ylabel(r"$\max_j |u_j - \sin\pi x_j|$")
ax1.set_title("1c: Fehler über $\\Delta x$")
ax1.legend(fontsize=8)

N = 63
for b in ["reduce", "mirror"]:
    x, u = solve(f, N, 6, b)
    ax2.semilogy(x, np.abs(u - np.sin(np.pi * x)), styles[b], ms=3, label=f"Ordnung 6, {b}")
ax2.set_xlabel("$x$")
ax2.set_ylabel(r"$|u_j - \sin\pi x_j|$")
ax2.set_title(f"Fehler über $x$, $N = {N}$")
ax2.legend(fontsize=8)
fig.tight_layout()
plt.show()

> **Frage:** Welche Ordnungen bekommt man?
>
> **Antwort:** Mit Spiegelung am Rand genau die Ordnung des Sterns: 2, 4 und 6.
> Mit `"reduce"` ergibt Ordnung 4 weiter etwa 4 (die Raten nähern sich von oben, 4,7 → 4,1), Ordnung 6 aber nur **5**.
> Die randnahen Zeilen haben einen größeren Abbruchfehler $\tau$, und ein Fehler $\tau$ in einer Zeile nahe
> $x = 0$ verschiebt die Lösung um etwa $\Delta x^2\tau$ (Zeile von $A^{-1}$ mal Zeilenzahl). Für $\sin\pi x$ ist
> $\tau_1 = \frac{\Delta x^2}{12}u''''(x_1) \approx \frac{\pi^5}{12}\Delta x^3$, weil $u''''(0) = 0$; das gibt
> einen Beitrag $\sim\Delta x^5$. Bei Ordnung 4 ist der kleiner als der innere Fehler $\Delta x^4$, bei Ordnung 6 größer.
> Rechts sieht man es: bei `"reduce"` ist der Fehler auch in der Mitte größer, die schwache Randzeile wirkt überall.
>
> **Frage:** Was macht man am Rand?
>
> **Antwort:** Entweder den Stern in den randnahen Zeilen verkleinern (einfach, kostet hier eine Ordnung bei 6),
> Geisterpunkte aus der Randbedingung bestimmen (hier per Spiegelung, nur richtig für ungerade fortsetzbare Lösungen)
> oder einseitige Sterne gleicher Ordnung verwenden (allgemein, aber $A$ wird unsymmetrisch).
>
> Je höher die Ordnung, desto früher ist man am Rundungsboden: Ordnung 6 mit Spiegelung erreicht $2\cdot10^{-13}$
> schon bei $N = 127$. Danach wachsen alle Kurven gleich an, der Rundungsfehler hängt nicht vom Stern ab.

### 1d) Neumann-Rand $u'(1) = 0$

Neues Testproblem: $f = \frac{\pi^2}{4}\sin\frac{\pi x}{2}$, exakte Lösung $u = \sin\frac{\pi x}{2}$
(es gilt $u(0) = 0$ und $u'(1) = \frac{\pi}{2}\cos\frac{\pi}{2} = 0$).

Jetzt ist $u(1) = u_{N+1}$ unbekannt und bekommt eine eigene Zeile. Zwei Varianten in `solve_neumann(f, N, bc)`:

- `"first"`: einseitige Differenz $\frac{u_{N+1} - u_N}{\Delta x} = 0$, Abbruchfehler $O(\Delta x)$.
- `"ghost"`: Geisterpunkt aus der zentralen Differenz $\frac{u_{N+2} - u_N}{2\Delta x} = 0$, also $u_{N+2} = u_N$.
  Eingesetzt in den Stern am Punkt $N+1$: $\frac{-2u_N + 2u_{N+1}}{\Delta x^2} = f(1)$, halbiert, damit $A$ symmetrisch bleibt.

In [ ]:
fn = lambda x: np.pi**2 / 4 * np.sin(np.pi * x / 2)
un = lambda x: np.sin(np.pi * x / 2)
ks = np.arange(2, 17)
Ns = 2**ks - 1
dxs = 1 / (Ns + 1)
errs_d = {}
for bc in ["first", "ghost"]:
    errs_d[bc] = np.array([np.max(np.abs(u - un(x))) for x, u in (solve_neumann(fn, N, bc) for N in Ns)])
    rates = np.log2(errs_d[bc][:-1] / errs_d[bc][1:])
    print(f"{bc:6s} Raten N = 3 ... 4095: " + " ".join(f"{r:.2f}" for r in rates[:11]))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
for bc, st in [("first", "o-"), ("ghost", "s--")]:
    ax1.loglog(dxs, errs_d[bc], st, ms=4, label=f'"{bc}"')
for p in [1, 2]:
    ax1.loglog(dxs[:6], 1e-3 * (dxs[:6] / dxs[0])**p, "k-", lw=0.6)
    ax1.text(dxs[5], 1e-3 * (dxs[5] / dxs[0])**p, f"  $\\Delta x^{p}$", va="center", fontsize=8)
ax1.set_xlabel(r"$\Delta x$")
ax1.set_ylabel(r"$\max_j |u_j - \sin\frac{\pi x_j}{2}|$")
ax1.set_title("1d: Fehler über $\\Delta x$")
ax1.legend()

N = 15
dx = 1 / (N + 1)
for bc, st in [("first", "o-"), ("ghost", "s--")]:
    x, u = solve_neumann(fn, N, bc)
    ax2.plot(x, u - un(x), st, ms=4, label=f'"{bc}"')
ax2.plot(x, -np.pi**2 / 8 * dx * x, "k:", lw=1, label=r"$-\frac{\pi^2}{8}\Delta x\, x$")
ax2.axhline(0, color="gray", lw=0.5)
ax2.set_xlabel("$x$")
ax2.set_ylabel(r"$u_j - \sin\frac{\pi x_j}{2}$")
ax2.set_title(f"Fehler über $x$, $N = {N}$")
ax2.legend()
fig.tight_layout()
plt.show()

> **Frage:** Welche Steigung bekommt man?
>
> **Antwort:** Mit der einseitigen Differenz (`"first"`) Steigung **1**, mit dem Geisterpunkt (`"ghost"`) Steigung **2**.
> Alle inneren Zeilen sind in beiden Fällen zweiter Ordnung, die eine Randzeile entscheidet.
>
> **Frage:** Wie wirkt sich die Diskretisierung des Randes auf die ganze Lösung aus?
>
> **Antwort:** Der Fehler bleibt nicht am Rand, er steckt im ganzen Intervall. Bei `"first"` ist er eine Gerade
> $-\frac{\pi^2}{8}\Delta x\,x$: Taylor gibt $\frac{u_{N+1} - u_N}{\Delta x} = u'(1) - \frac{\Delta x}{2}u''(1)$, die Zeile erzwingt
> also in Wahrheit $u'(1) = \frac{\Delta x}{2}u''(1) = -\frac{\pi^2}{8}\Delta x$. Der falsche Fluss am Rand addiert die Lösung
> von $-u'' = 0$ mit $u(0) = 0$ und dieser Steigung, und das ist eine Gerade durch den Ursprung.
> Eine elliptische Gleichung verteilt jeden Randfehler über das ganze Gebiet.

### 1e) 2D-Poisson

$-\Delta u = 2\pi^2\sin\pi x\,\sin\pi y$ auf dem Einheitsquadrat, $u = 0$ am Rand, exakt $u = \sin\pi x\,\sin\pi y$.
$N\times N$ innere Punkte, also $N^2$ Unbekannte. Die Matrix entsteht aus der 1D-Matrix über das Kronecker-Produkt:

$$A_{2D} = A\otimes I + I\otimes A \quad\Rightarrow\quad \frac{4u_{ij} - u_{i\pm1,j} - u_{i,j\pm1}}{\Delta x^2}\ \text{(5-Punkte-Stern)}$$

`laplacian_2d` und `solve_2d` in `fd.py`. Die rechte Seite wird mit `meshgrid(..., indexing="ij")` und `.ravel()` in
derselben Reihenfolge zum Vektor wie die Unbekannten.

In [ ]:
import time

from fd import laplacian_2d, solve_2d

f2 = lambda x, y: 2 * np.pi**2 * np.sin(np.pi * x) * np.sin(np.pi * y)
ks = np.arange(2, 11)  # N = 1023 takes ~10 s
Ns = 2**ks - 1
errs, times = [], []
for N in Ns:
    t0 = time.perf_counter()
    X, Y, U = solve_2d(f2, N)
    times.append(time.perf_counter() - t0)
    errs.append(np.max(np.abs(U - np.sin(np.pi * X) * np.sin(np.pi * Y))))
errs, times = np.array(errs), np.array(times)
slope_e = np.polyfit(np.log(1 / (Ns + 1)), np.log(errs), 1)[0]
slope_t = np.polyfit(np.log(Ns[-4:]), np.log(times[-4:]), 1)[0]
print(f"Steigung Fehler über dx: {slope_e:.2f},  Laufzeit über N (letzte 4): {slope_t:.2f}")
print("     N   Unbekannte    Fehler      Zeit   nnz(A)")
for N, e, t in zip(Ns, errs, times):
    print(f"{N:6d} {N * N:11d}  {e:.2e}  {t:7.3f} s  {laplacian_2d(N).nnz:9d}")

fig, axs = plt.subplots(1, 3, figsize=(15, 4.3))
Xs, Ys, Us = solve_2d(f2, 31)
im = axs[0].pcolormesh(Xs, Ys, Us, shading="auto")
fig.colorbar(im, ax=axs[0])
axs[0].set_aspect("equal")
axs[0].set_title("Lösung $N = 31$")
axs[0].set_xlabel("$x$")
axs[0].set_ylabel("$y$")
axs[1].loglog(1 / (Ns + 1), errs, "o-", ms=4, label="gemessen")
axs[1].loglog(1 / (Ns + 1), np.pi**2 / 12 / (Ns + 1) ** 2, "k--", lw=1, label=r"$\pi^2\Delta x^2/12$")
axs[1].set_xlabel(r"$\Delta x$")
axs[1].set_title("1e: Fehler")
axs[1].legend()
axs[2].loglog(Ns, times, "o-", ms=4, label="spsolve")
for p, ls in [(2, ":"), (3, "--")]:
    axs[2].loglog(Ns, times[-1] * (Ns / Ns[-1]) ** p, "k" + ls, lw=1, label=f"$\\propto N^{p}$")
axs[2].set_xlabel("$N$ (pro Richtung)")
axs[2].set_ylabel("Zeit / s")
axs[2].set_title("1e: Laufzeit")
axs[2].legend()
fig.tight_layout()
plt.show()

> **Frage:** Wie skalieren Fehler und Laufzeit mit $N$?
>
> **Antwort:** Der Fehler fällt mit Steigung 2,00 über $\Delta x$, also wie $N^{-2}$, genau wie in 1D. Er liegt exakt
> auf $\pi^2\Delta x^2/12$: Die rechte Seite ist ein Eigenvektor von $A_{2D}$ zum Eigenwert $2\lambda_1$, und
> $\lambda_1 \approx \pi^2(1 - \pi^2\Delta x^2/12)$. Bis $N = 1023$ ($\Delta x \approx 10^{-3}$) wächst er nicht wieder an,
> das Rundungsminimum aus 1b ($\Delta x \approx 1{,}5\cdot10^{-5}$) wäre in 2D mit $4\cdot10^9$ Unbekannten unerreichbar.
>
> Die Laufzeit wächst für große $N$ mit Steigung $\approx 2{,}8$, also fast wie $N^3 = n^{1{,}5}$ mit $n = N^2$
> Unbekannten. Ein feineres Gitter (halbes $\Delta x$) kostet damit etwa achtmal so viel Zeit, viertelt aber nur den
> Fehler: $N = 511$ braucht 1,3 s, $N = 1023$ schon 10 s. In 1D wäre es $O(N)$. Grund: Die Matrix hat nur 5 Einträge pro
> Zeile, aber die Bandbreite $N$, und beim Faktorisieren füllt sich das Band auf (fill-in). SuperLU sortiert die
> Unbekannten um und landet so bei etwa $n^{1{,}5}$ statt $N^4$ für ein naives Band-Gauß. Für noch größere Gitter
> nimmt man iterative Löser (CG, Mehrgitter, Kapitel 4) oder für dieses Problem die FFT (Kapitel 3.9).

## 2 Zeitabhängige Wärmeleitung $u_t = u_{xx}$

### 2a) Linienmethode

Nur der Raum wird diskretisiert: $\dot u = -A u$ ist das lineare System $\dot x = Jx$ aus Exercise 3 mit $J = -A$.
Die Schritte aus `2026-10-28/ode.py` werden unverändert importiert, neu in `fd.py` sind nur
`implicit_euler_step` (sparse) und `crank_nicolson_step`.

In [ ]:
import sys

sys.path.insert(0, "../2026-10-28")  # Exercise 3, used unchanged
from ode import critical_dt, euler_step, heun_step, integrate, order, rk4_step, stability_function

from fd import crank_nicolson_step, implicit_euler_step, pulse_series

N = 50
x, dx = grid(N)
J = -laplacian(N)  # u' = -A u, same form x' = J x as in Exercise 3
u0 = np.sin(np.pi * x)
t_end = 0.1
u_pde = np.exp(-np.pi**2 * t_end) * u0
methods = {"expl. Euler": euler_step, "impl. Euler": implicit_euler_step, "Crank-Nicolson": crank_nicolson_step}

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(x, u0, "k:", label="$t = 0$")
ax.plot(x, u_pde, "k-", lw=3, alpha=0.3, label="exakt, $t = 0{,}1$")
for name, step in methods.items():
    t, U = integrate(step, J, u0, 1e-4, t_end)
    print(f"{name:15s} dt = 1e-4:  max|u - u_PDE| = {np.max(np.abs(U[-1] - u_pde)):.2e}")
    ax.plot(x, U[-1], "--", label=name)
ax.set_xlabel("$x$")
ax.set_ylabel("$u$")
ax.set_title(r"2a: Linienmethode, $N = 50$, $\Delta t = 10^{-4}$")
ax.legend()
fig.tight_layout()
plt.show()

> **Frage:** Welche Teile des Codes aus Exercise 3 kann man unverändert verwenden?
>
> **Antwort:** `euler_step`, `heun_step`, `rk4_step` brauchen nur `J @ x` und laufen mit der sparse-Matrix
> $J = -A$ unverändert, ebenso `integrate`, `order`, `stability_function` und sogar `critical_dt`: Das rechnet
> mit dem größeren der beiden übergebenen Eigenwerte, also gibt man $\lambda_1, \lambda_N$ statt $k_1, k_2$.
> Nur `implicit_euler_step` passt nicht, weil es `np.eye` und `np.linalg.solve` (dense) benutzt. In `fd.py` steht
> deshalb eine Version mit `spsolve`. Crank–Nicolson ist dort ein halber expliziter Schritt, gefolgt von einem
> halben impliziten Schritt: $(I - \tfrac{\Delta t}{2}J)\,u^{n+1} = u^n + \tfrac{\Delta t}{2}Ju^n$.

### 2b) Genauigkeit

Zwei Referenzen bei $t = 0{,}1$: die PDE-Lösung $e^{-\pi^2 t}\sin\pi x_j$ und die exakte ODE-Lösung
$e^{-\lambda_1 t}\sin\pi x_j$ (der Start ist Eigenvektor von $A$). Der Abstand der beiden ist der reine Raumfehler.

In [ ]:
lam1 = 4 / dx**2 * np.sin(np.pi * dx / 2) ** 2
lamN = 4 / dx**2 * np.sin(N * np.pi * dx / 2) ** 2
u_ode = np.exp(-lam1 * t_end) * u0  # exact solution of the ODE system (u0 is an eigenvector)
print(f"expl. Euler stabil nur für dt < 2/lambda_N = {2 / lamN:.2e}")
print(f"lambda_1 = {lam1:.4f} (pi^2 = {np.pi**2:.4f}),  Raumfehler |u_ODE - u_PDE| = {np.max(np.abs(u_ode - u_pde)):.2e}")

dts = t_end / 2 ** np.arange(1, 13)
fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
for name, step in methods.items():
    e_pde, e_ode = [], []
    for dt in dts:
        with np.errstate(over="ignore", invalid="ignore"):
            u = integrate(step, J, u0, dt, t_end)[1][-1]
        e_pde.append(np.max(np.abs(u - u_pde)))
        e_ode.append(np.max(np.abs(u - u_ode)))
    e_pde, e_ode = np.array(e_pde), np.array(e_ode)
    ok = dts < 2 / lamN if step is euler_step else dts > 0  # explicit Euler only below its limit (2c)
    print(f"{name:15s} Steigung gegen u_ODE: {order(dts[ok][-4:], e_ode[ok][-4:]):.2f}")
    axs[0].loglog(dts[ok], e_ode[ok], "o-", ms=4, label=name)
    axs[1].loglog(dts[ok], e_pde[ok], "o-", ms=4, label=name)
for ax, ref in zip(axs[:2], ["ODE-Lösung (nur Zeitfehler)", "PDE-Lösung (Zeit + Raum)"]):
    ax.set_xlabel(r"$\Delta t$")
    ax.set_title(f"Fehler gegen {ref}, $N = 50$")
axs[1].axhline(np.max(np.abs(u_ode - u_pde)), color="gray", ls=":", label="Raumfehler $N = 50$")
axs[0].set_ylabel(r"$\max_j|u_j - u_{\mathrm{ref}}|$ bei $t = 0{,}1$")
axs[0].legend()
axs[1].legend()

Ns = 2 ** np.arange(3, 9) - 1
for name, step in [("impl. Euler", implicit_euler_step), ("Crank-Nicolson", crank_nicolson_step)]:
    errs = []
    for n in Ns:
        xn = grid(n)[0]
        u = integrate(step, -laplacian(n), np.sin(np.pi * xn), 1e-4, t_end)[1][-1]
        errs.append(np.max(np.abs(u - np.exp(-np.pi**2 * t_end) * np.sin(np.pi * xn))))
    print(f"{name:15s} Fehler über dx (dt = 1e-4): " + "  ".join(f"{e:.1e}" for e in errs))
    axs[2].loglog(1 / (Ns + 1), errs, "o-", ms=4, label=name)
axs[2].loglog(1 / (Ns + 1), np.pi**4 * t_end * np.exp(-np.pi**2 * t_end) / 12 / (Ns + 1) ** 2, "k--", lw=1,
              label=r"Raumfehler $\propto\Delta x^2$")
axs[2].set_xlabel(r"$\Delta x$")
axs[2].set_title(r"Fehler gegen PDE-Lösung, $\Delta t = 10^{-4}$")
axs[2].legend()
fig.tight_layout()
plt.show()

> **Frage:** Welche Steigungen erwartet man?
>
> **Antwort:** In $\Delta t$ gegen die ODE-Lösung: expliziter und impliziter Euler 1, Crank–Nicolson 2. Gemessen:
> 1,00 / 1,00 / 2,00. In $\Delta x$: 2 (3-Punkte-Stern). Crank–Nicolson mit $\Delta t = 10^{-4}$ trifft die Gerade
> $\Delta x^2$ genau (Fehler viertelt sich pro Halbierung, bis $4{,}6\cdot10^{-6}$).
>
> **Frage:** Warum hört der Fehler irgendwann auf zu fallen?
>
> **Antwort:** Gegen die PDE-Lösung bleibt immer der Fehler der anderen Diskretisierung übrig. In der
> $\Delta t$-Studie ist das der Raumfehler von $N = 50$, $1{,}16\cdot10^{-4}$ (graue Linie). In der
> $\Delta x$-Studie bleibt der impliziten Euler bei $\approx 1{,}9\cdot10^{-4}$ stehen, das ist sein Zeitfehler bei
> $\Delta t = 10^{-4}$. Crank–Nicolson hat dort nur $\sim10^{-9}$ Zeitfehler und fällt deshalb weiter.
>
> Zwei Auffälligkeiten im mittleren Bild: Der explizite Euler liegt *unter* dem Raumfehler, und Crank–Nicolson hat
> bei $\Delta t \approx 6\cdot10^{-3}$ einen tiefen Knick. Das ist Fehlerauslöschung: Wegen $\lambda_1 < \pi^2$ klingt
> die ODE-Lösung etwas zu langsam ab, expliziter Euler und Crank–Nicolson dämpfen pro Schritt etwas zu stark. Die
> Fehler haben entgegengesetzte Vorzeichen und heben sich teilweise auf, Crank–Nicolson an einer Stelle fast ganz.
> Der implizite Euler dämpft zu schwach, sein Fehler addiert sich zum Raumfehler.
>
> Der explizite Euler ist nur für $\Delta t < 2/\lambda_N = 1{,}92\cdot10^{-4}$ eingezeichnet. Darüber explodiert er,
> allerdings verzögert: Der Start enthält nur Mode 1, die hohen Moden kommen erst über Rundungsfehler ($10^{-16}$)
> hinein. Bei $\Delta t = 0{,}05$ (2 Schritte, Faktor 519 pro Schritt) erreicht die höchste Mode erst
> $3\cdot10^{-12}$, bei $\Delta t = 3{,}9\cdot10^{-4}$ (256 Schritte, Faktor 3,1) schon $10^{107}$.

### 2c) Stabilität

Eigenwerte $\lambda_k = \frac{4}{\Delta x^2}\sin^2\frac{k\pi\Delta x}{2}$, Eigenvektoren $\sin(k\pi x_j)$. Mode $k$ wird
pro Schritt mit $R(z_k)$, $z_k = -\lambda_k\Delta t$, multipliziert.

In [ ]:
N = 20
x20, dx20 = grid(N)
lam, V = np.linalg.eigh(laplacian(N).toarray())  # ascending: lam[0] = lambda_1, lam[-1] = lambda_N
k = np.arange(1, N + 1)
print("Eigenwerte gegen Formel 4/dx^2 sin^2(k pi dx/2):", np.allclose(lam, 4 / dx20**2 * np.sin(k * np.pi * dx20 / 2) ** 2))

fig, axs = plt.subplots(1, 2, figsize=(13, 4.8))
xs = np.r_[0, x20, 1]
for i in [0, 1, 2, N - 1]:
    v = V[:, i] * np.sign(V[0, i])  # eigh returns arbitrary signs
    axs[0].plot(xs, np.r_[0, v, 0], "o-", ms=3, label=f"$k = {i + 1}$, $\\lambda = {lam[i]:.0f}$")
axs[0].set_xlabel("$x$")
axs[0].set_title(f"Eigenvektoren von $A$, $N = {N}$")
axs[0].legend(fontsize=8)

re, im = np.meshgrid(np.linspace(-4, 2.2, 497), np.linspace(-3.5, 3.5, 401))
z = re + 1j * im
regions = {"expl. Euler": stability_function(euler_step, z), "Heun": stability_function(heun_step, z),
           "RK4": stability_function(rk4_step, z), "impl. Euler": 1 / (1 - z), "Crank-Nicolson": (1 + z / 2) / (1 - z / 2)}
for (name, R), c in zip(regions.items(), ["C0", "C1", "C2", "C3", "C4"]):
    axs[1].contour(re, im, np.abs(R), levels=[1], colors=c, linestyles="-" if name != "Crank-Nicolson" else "--")
    axs[1].plot([], [], color=c, label=name)
dt = 1e-3
axs[1].plot(-lam * dt, 0 * lam, "kx", label=rf"$z_k = -\lambda_k\Delta t$, $\Delta t = 10^{{-3}}$")
axs[1].axhline(0, color="gray", lw=0.5)
axs[1].axvline(0, color="gray", lw=0.5)
axs[1].set_aspect("equal")
axs[1].set_xlabel(r"Re $z$")
axs[1].set_ylabel(r"Im $z$")
axs[1].set_title(r"Stabilitätsränder $|R(z)| = 1$")
axs[1].legend(fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()

print("\n  N   lambda_N   dt_krit Euler  von Neumann dx^2/2   Heun       RK4")
for n in [10, 20, 40, 80]:
    d = grid(n)[1]
    lN = 4 / d**2 * np.sin(n * np.pi * d / 2) ** 2
    l1 = 4 / d**2 * np.sin(np.pi * d / 2) ** 2
    print(f"{n:3d}  {lN:9.1f}   {critical_dt(euler_step, l1, lN):.3e}      {d**2 / 2:.3e}       "
          f"{critical_dt(heun_step, l1, lN):.3e}  {critical_dt(rk4_step, l1, lN):.3e}")

In [ ]:
# Just above the limit: explicit Euler, N = 20, dt = 1.02 * dt_krit
dt_k = critical_dt(euler_step, lam[0], lam[-1])
J20 = -laplacian(N)
t, U = integrate(euler_step, J20, np.sin(np.pi * x20), 1.02 * dt_k, 1.0)
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
for n in np.searchsorted(t, [0.6, 0.8, 0.9, 1.0]):
    axs[0].plot(xs, np.r_[0, U[n], 0], "o-", ms=3, label=f"$t = {t[n]:.2f}$")
axs[0].set_xlabel("$x$")
axs[0].set_title(r"expl. Euler, $\Delta t = 1{,}02\,\Delta t_{\mathrm{krit}}$, $N = 20$")
axs[0].legend(fontsize=8)
modes = np.abs(U @ V)  # amplitude of each eigenmode over time
axs[1].semilogy(t, modes[:, 0], label="Grundmode $k = 1$")
axs[1].semilogy(t, modes[:, -1], label="höchste Mode $k = N$")
axs[1].semilogy(t, np.exp(-lam[0] * t) * modes[0, 0], "k:", lw=1, label=r"exakt $e^{-\lambda_1 t}$")
axs[1].set_xlabel("$t$")
axs[1].set_title("Amplitude der Eigenmoden")
axs[1].legend(fontsize=8)
fig.tight_layout()
plt.show()
print(f"|R(z_N)| = {abs(1 - lam[-1] * 1.02 * dt_k):.3f} pro Schritt, {len(t) - 1} Schritte")

> **Frage:** Welcher Eigenwert entscheidet über die Stabilität?
>
> **Antwort:** Der größte, $\lambda_N \approx 4/\Delta x^2$. Alle $z_k$ liegen auf der negativen reellen Achse, $z_N$ am
> weitesten links. Im Bild mit $\Delta t = 10^{-3}$ und $N = 20$ ist $z_N = -1{,}75$: noch in den Gebieten von Euler
> ($[-2, 0]$), Heun ($[-2, 0]$) und RK4 ($[-2{,}785, 0]$). Die Tabelle zeigt $\Delta t_{\text{krit}} = 2/\lambda_N$
> (Euler, Heun) bzw. $2{,}785/\lambda_N$ (RK4). Die Von-Neumann-Schranke $\Delta x^2/2$ liegt immer knapp darunter
> ($4{,}13$ statt $4{,}22\cdot10^{-3}$ bei $N = 10$) und nähert sich mit feinerem Gitter an ($7{,}621$ statt
> $7{,}624\cdot10^{-5}$ bei $N = 80$). Grund: von Neumann nimmt $\sin^2 = 1$, auf dem Gitter ist es knapp darunter.
> Halbes $\Delta x$ heißt ein Viertel $\Delta t$.
>
> **Frage:** Wie sieht die Lösung knapp über dem kritischen Zeitschritt aus?
>
> **Antwort:** Bei $1{,}02\,\Delta t_{\text{krit}}$ wird die höchste Mode pro Schritt mit $|R| = 1{,}04$ multipliziert
> und wechselt jedes Mal das Vorzeichen. Sie wächst von Rundungsniveau exponentiell, während die Grundmode richtig
> abklingt. Bei $t \approx 0{,}87$ (rund 750 Schritte) überholt sie die Grundmode. Danach sieht man nur noch einen
> Zickzack von Punkt zu Punkt mit sinusförmiger Einhüllender, genau den Eigenvektor $k = N$ aus dem linken Bild.
>
> **Frage:** Warum sind impliziter Euler und Crank–Nicolson für jedes $\Delta t$ stabil?
>
> **Antwort:** Für $z < 0$ gilt $|1/(1-z)| < 1$ und $|(1+z/2)/(1-z/2)| < 1$, egal wie groß $|z|$ ist. Das
> Stabilitätsgebiet enthält die ganze linke Halbebene (A-stabil): Beim impliziten Euler ist alles außerhalb des roten
> Kreises stabil, bei Crank–Nicolson genau die linke Halbebene (gestrichelt auf der imaginären Achse).

### 2d) Steifheit und unglatte Startwerte

Rechteckpuls $u(x,0) = 1$ für $0{,}25 < x < 0{,}75$, $N = 99$ ($\Delta x = 0{,}01$). Referenzen: die Sinusreihe
`pulse_series` (PDE) und die Eigenzerlegung von $A$ (ODE): $u(t) = V e^{-\Lambda t} V^T u_0$.

In [ ]:
N = 99
x99, dx99 = grid(N)
A99 = laplacian(N)
lam, V = np.linalg.eigh(A99.toarray())
u0 = ((x99 > 0.25) & (x99 < 0.75)).astype(float)
t_end = 0.1
u_pde = pulse_series(x99, t_end, kmax=200)
u_ode = V @ (np.exp(-lam * t_end) * (V.T @ u0))  # exact ODE solution via eigen decomposition
print(f"lambda_1 = {lam[0]:.2f}, lambda_N = {lam[-1]:.0f}, Verhältnis {lam[-1] / lam[0]:.0f}")
print(f"Raumfehler |u_ODE - u_PDE| bei t = 0,1: {np.max(np.abs(u_ode - u_pde)):.1e}")

steppers = {"impl. Euler": implicit_euler_step, "Crank-Nicolson": crank_nicolson_step}
dts = t_end / 2 ** np.arange(0, 11)
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
for name, step in steppers.items():
    e = [np.max(np.abs(integrate(step, -A99, u0, dt, t_end)[1][-1] - u_ode)) for dt in dts]
    print(f"{name:15s} Fehler gegen u_ODE: " + " ".join(f"{v:.0e}" for v in e))
    axs[0].loglog(dts, e, "o-", ms=4, label=name)
axs[0].axvline(0.01, color="gray", lw=0.8)
axs[0].set_xlabel(r"$\Delta t$")
axs[0].set_title(r"2d: Fehler gegen ODE-Lösung bei $t = 0{,}1$, $N = 99$")
axs[0].legend()

xs = np.r_[0, x99, 1]
axs[1].plot(xs, np.r_[0, u0, 0], "k:", lw=1, label="Start")
axs[1].plot(xs, np.r_[0, u_ode, 0], "k-", lw=3, alpha=0.3, label="exakt (ODE)")
for name, step in steppers.items():
    axs[1].plot(xs, np.r_[0, integrate(step, -A99, u0, 0.01, t_end)[1][-1], 0], ".-", ms=3, label=name)
axs[1].set_xlabel("$x$")
axs[1].set_title(r"$t = 0{,}1$ nach 10 Schritten mit $\Delta t = 0{,}01$ ($r = 100$)")
axs[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
z = np.linspace(-1000, 0, 4001)
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
axs[0].plot(z, np.abs(1 / (1 - z)), label="impl. Euler $1/(1-z)$")
axs[0].plot(z, np.abs((1 + z / 2) / (1 - z / 2)), label=r"Crank-Nicolson $\frac{1+z/2}{1-z/2}$")
axs[0].plot(z, np.exp(z), "k--", label="exakt $e^z$")
axs[0].axvline(-lam[-1] * 0.01, color="gray", lw=0.8, label=r"$z_N = -\lambda_N\cdot 0{,}01$")
axs[0].set_xlabel("$z$")
axs[0].set_ylabel("$|R(z)|$")
axs[0].set_title("Dämpfung pro Schritt")
axs[0].legend(fontsize=8)

vN = V[:, -1]
for name, step in steppers.items():
    t, U = integrate(step, -A99, u0, 0.01, 1.0)
    axs[1].semilogy(t, np.abs(U @ vN), "o-", ms=2, label=f"{name}, höchste Mode")
axs[1].semilogy(t, np.abs(vN @ u0) * np.exp(-lam[-1] * t) + 1e-300, "k--", label="exakt, höchste Mode")
axs[1].semilogy(t, np.abs(V[:, 0] @ u0) * np.exp(-lam[0] * t), "k:", label="exakt, Grundmode")
axs[1].set_ylim(1e-20, 10)
axs[1].set_xlabel("$t$")
axs[1].set_title(r"Amplitude der Moden, $\Delta t = 0{,}01$")
axs[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"R_CN(z_N) = {(1 - lam[-1] * 0.005) / (1 + lam[-1] * 0.005):.4f},  R_IE(z_N) = {1 / (1 + lam[-1] * 0.01):.2e}")
print("\nExpliziter Euler bis t = 0,1:")
for n in [99, 199, 399]:
    d = grid(n)[1]
    lN = 4 / d**2 * np.sin(n * np.pi * d / 2) ** 2
    print(f"  N = {n:3d}: dt_krit = {2 / lN:.2e}, mindestens {int(np.ceil(t_end * lN / 2))} Schritte (statt 10)")

> **Frage:** Warum ist Crank–Nicolson, in 2b das beste Verfahren, jetzt schlechter als der implizite Euler?
>
> **Antwort:** Der Sprung enthält alle Moden, die Eigenwerte reichen von 9,87 bis 39 990 (Verhältnis 4052, steif).
> Bei $\Delta t = 0{,}01$ ist $z_N = -400$ und $R_{\text{CN}}(z_N) = -0{,}990$: Die schnellen Moden werden kaum gedämpft
> und wechseln jeden Schritt das Vorzeichen. Exakt wäre $e^{-400} \approx 0$. Der implizite Euler hat
> $R_{\text{IE}}(z_N) = 0{,}0025$, er dämpft sie fast sofort weg wie die echte Lösung (L-stabil: $R(z) \to 0$ für
> $z \to -\infty$, bei CN dagegen $\to -1$). Gemessen bei $t = 0{,}1$: Fehler 0,016 (IE) gegen 0,33 (CN), mit
> Überschwingern direkt an den Sprungstellen. Crank–Nicolson ist erst unter $\Delta t \approx 1{,}6\cdot10^{-3}$
> wieder besser, dann aber mit Ordnung 2.
> (Der Raumfehler ist hier $5\cdot10^{-3}$ statt $10^{-4}$: Die Sprünge liegen genau auf Gitterpunkten und $u_0$ ist dort
> 0, der diskrete Puls ist also eine Zelle schmaler.)
>
> **Frage:** Verschwinden die schnellen Moden überhaupt?
>
> **Antwort:** Bei Crank–Nicolson nur geometrisch: Die höchste Mode fällt in 100 Schritten von 0,10 auf 0,037
> ($0{,}99^{100} \approx 0{,}37$). Ab $t \approx 0{,}47$ ist sie größer als die Grundmode, die Lösung ist dann vor allem
> Zickzack. Beim impliziten Euler ist sie nach 5 Schritten auf Rundungsniveau.
>
> **Frage:** Wie viele Schritte bräuchte der explizite Euler, und wie ändert sich das bei feinerem Gitter?
>
> **Antwort:** $\Delta t_{\text{krit}} = 2/\lambda_N = 5\cdot10^{-5}$, also mindestens 2000 Schritte bis $t = 0{,}1$
> statt 10. Halbes $\Delta x$ heißt viermal so viele Schritte (8000 bei $N = 199$, 32 000 bei $N = 399$), und jeder
> Schritt kostet doppelt so viel: Gesamtaufwand $\propto N^3$. Deshalb rechnet man Diffusion implizit.